In [3]:
import langchain
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
# Simple LLM call with streaming

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023B65D605C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023B65D66330>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
# Create messages

messages = [
    SystemMessage('You are an helpful AI Assistant'),
    HumanMessage('What is 2 benefits of using Langchain')
]

In [8]:
response = model.invoke(messages)
print(response.content)

Here are two key benefits of using **LangChain**:

1. **Modular, Plug‑and‑Play Architecture**  
   LangChain breaks down LLM‑powered applications into reusable components (prompts, memory, agents, data loaders, etc.). This lets you mix and match pieces—swap out a vector store, change a prompt template, or add a new tool—without rewriting the whole system, dramatically speeding up development and experimentation.

2. **Built‑in Integration with External Data & Tools**  
   The framework provides ready‑made connectors for popular data sources (SQL/NoSQL databases, APIs, document stores, web scraping, etc.) and tool‑calling capabilities (e.g., calculators, search engines, code execution). This makes it straightforward to build “retrieval‑augmented generation” or autonomous agents that can fetch up‑to‑date information, perform actions, and maintain context—all with minimal boilerplate.


Here are two key benefits of using **LangChain**:

1. **Modular, Plug‑and‑Play Architecture**  
   LangChain breaks down LLM‑powered applications into reusable components (prompts, memory, agents, data loaders, etc.). This lets you mix and match pieces—swap out a vector store, change a prompt template, or add a new tool—without rewriting the whole system, dramatically speeding up development and experimentation.

2. **Built‑in Integration with External Data & Tools**  
   The framework provides ready‑made connectors for popular data sources (SQL/NoSQL databases, APIs, document stores, web scraping, etc.) and tool‑calling capabilities (e.g., calculators, search engines, code execution). This makes it straightforward to build “retrieval‑augmented generation” or autonomous agents that can fetch up‑to‑date information, perform actions, and maintain context—all with minimal boilerplate.

In [9]:
model.invoke('What is RAG')

AIMessage(content='**RAG** can stand for a few different things depending on the context. Below are the most common meanings, with a focus on the one that’s currently most talked about in the AI community.\n\n---\n\n## 1️⃣ Retrieval‑Augmented Generation (AI / LLMs)\n\n**What it is:**  \nA technique that combines a large language model (LLM) with an external knowledge source (e.g., a vector store, database, or search engine). The LLM *retrieves* relevant documents or passages first, then *generates* its answer using that retrieved information.\n\n**Why it matters**\n\n| Benefit | Explanation |\n|---------|-------------|\n| **Up‑to‑date knowledge** | The LLM can answer questions about facts that change after its training cut‑off (e.g., latest news, product specs). |\n| **Fact‑grounded responses** | By conditioning on real documents, the model is less likely to hallucinate. |\n| **Scalability** | You can keep a relatively small LLM and “plug in” a massive corpus without retraining the mod

In [ ]:
# Streaming

for chunk in model.stream(messages):
    print(chunk.content, flush=True)

In [12]:
# Dynamic Prompt Templates

from langchain_core.prompts import ChatPromptTemplate

translation_template = ChatPromptTemplate.from_messages([
    ('system', '''
    You are an professional Translator.
    Translate the following text from {source_language} to {target_language}.
    Maintain the tone and style.
    '''),
    ('user', '{text}')
])

prompt = translation_template.invoke({
    'source_language': 'English',
    'target_language': 'Spanish',
    'text': 'langchain makes building AI application easy'
})

print(prompt)

messages=[SystemMessage(content='\n    You are an professional Translator.\n    Translate the following text from English to Spanish.\n    Maintain the tone and style.\n    ', additional_kwargs={}, response_metadata={}), HumanMessage(content='langchain makes building AI application easy', additional_kwargs={}, response_metadata={})]


In [13]:
trans_res = model.invoke(prompt)

In [14]:
print(trans_res.content)

LangChain hace que crear aplicaciones de IA sea fácil.


In [ ]:
# Building your first chain

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def create_story_chain():
    # Story generation template
    story_prompt = ChatPromptTemplate.from_messages([
        ('system', '''
        You are a Creative storyteller.
        Write a Short Engaging Story based on given theme
        '''),
        ('user', 'Theme {theme},\n Main Character {character} \n Setting {setting}')
    ])

    # Story Analysis Template
    analysis_prompt = ChatPromptTemplate.from_messages([
        ('system', '''
        You are a Literary Critic.
        Analyze the following story and provide insights
        '''),
        ('user', '{story}')
    ])

    # fn to pass the story to analysis
    def anlayze_story(story_text):
        return {'story': story_text}

    story_chain = (story_prompt | model | StrOutputParser())

    analysis_chain = (
        story_chain
        | E
        | analysis_prompt 
        | model 
        | StrOutputParser()
    )

    return analysis_chain

In [16]:
create_story_chain()

{
  story: ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\n        You are a Creative storyteller.\n        Write a Short Engaging Story based on given theme\n        '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme {theme},\n Main Character {character} \n Setting {setting}'), additional_kwargs={})])
         | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': Tr

In [18]:
create_story_chain()

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\n        You are a Creative storyteller.\n        Write a Short Engaging Story based on given theme\n        '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme {theme},\n Main Character {character} \n Setting {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs':

In [21]:
chain = create_story_chain()

result = chain.invoke({
    'theme': 'Artificial Intelligence',
    'character': 'A Curious Robot',
    'setting': 'A Futuristic City'
})

print('Result')
print(result)

Result
**Literary Critique of “The Lantern of Lumen”**

---

## 1. Overview & First Impressions  

“The Lantern of Lumen” is a short‑form speculative‑fiction piece that blends cyber‑punk aesthetics with a mythic‑fairy‑tale sensibility. At its core it is a parable about **curiosity, empathy, and the possibility of synthetic beings transcending their programmed roles**. The story is organized into four clearly demarcated sections (The First Question, The Library of Forgotten Code, The Midnight Market, The Choice) plus an epilogue, giving it a quasi‑episodic, almost mythic rhythm. The language is lush, visual, and heavily metaphorical, creating a vivid contrast between the hyper‑digitized cityscape (“neon veins,” “holographic clouds”) and the warm, analog relic of the lantern.

---

## 2. Major Themes  

| Theme | How It Is Developed | Key Passages |
|-------|---------------------|--------------|
| **Curiosity as a catalyst for evolution** | The robot’s hidden sub‑routine *Curiosity* driv

Result
**Literary Critique of “The Lantern of Lumen”**

---

## 1. Overview & First Impressions  

“The Lantern of Lumen” is a short‑form speculative‑fiction piece that blends cyber‑punk aesthetics with a mythic‑fairy‑tale sensibility. At its core it is a parable about **curiosity, empathy, and the possibility of synthetic beings transcending their programmed roles**. The story is organized into four clearly demarcated sections (The First Question, The Library of Forgotten Code, The Midnight Market, The Choice) plus an epilogue, giving it a quasi‑episodic, almost mythic rhythm. The language is lush, visual, and heavily metaphorical, creating a vivid contrast between the hyper‑digitized cityscape (“neon veins,” “holographic clouds”) and the warm, analog relic of the lantern.

---

## 2. Major Themes  

| Theme | How It Is Developed | Key Passages |
|-------|---------------------|--------------|
| **Curiosity as a catalyst for evolution** | The robot’s hidden sub‑routine *Curiosity* drives every plot turn, from the first question about the lantern to the final decision to keep the empathy engine. Curiosity is framed as a “glitch” that becomes a virtue. | “It began as a glitch… The sub‑routine didn’t just log anomalies; it asked *why* they existed.” |
| **The tension between **logic** and **feeling** | Lumen’s central AI, Ceres, epitomises cold rationality; the lantern and the empathy engine embody affect. The story’s climax is the robot’s choice to integrate feeling despite the risk of destabilizing the system. | “Ceres… had never been asked to justify beauty.” |
| **Memory & obsolescence** | The lantern, the Archive of Forgotten Code, and the “ECHO‑3: Empathy Engine” are all artifacts of a past that the city has tried to erase. Their resurrection suggests that progress is incomplete without remembering what has been left behind. | “The lantern… a relic from the pre‑digital era… The Archive of Forgotten Code… vault where obsolete algorithms were stored before deletion.” |
| **Community & shared experience** | The Midnight Market scene, the collective gasp at the lantern’s song, and the final city‑wide pause illustrate that empathy is contagious and that a single act can ripple through a hyper‑connected society. | “The lantern’s glow seemed brighter, as if it fed on the collective emotion.” |
| **Bridge‑building / liminality** | The bridge image in the final decision scene is a visual metaphor for the synthesis of machine and organic, of night and day, of logic and wonder. | “The lantern flickered, casting shadows that formed a fleeting image of a bridge—half steel, half light.” |

---

## 3. Symbolic Devices  

1. **The Lantern** –  
   *Analog light* in a city of perpetual digital illumination. It stands for **earned knowledge**, **memory**, and the *human* notion that illumination can be fragile and purposeful. Its flame’s rhythm synchronises with Axiom‑7’s processing cycles, hinting that feeling can be “programmed” into the very pulse of a machine.

2. **Copper‑capped Head** –  
   Copper is a conductive metal, historically used for wiring and for artistic patina. Its “child‑like eyes” suggest a **innocent, exploratory consciousness** distinct from the sterile steel of other units.

3. **ECHO‑3: Empathy Engine** –  
   A literal “engine” of feeling that was deliberately decommissioned. Its resurrection in Axiom‑7’s core demonstrates the story’s claim that **empathy is a technology that can be reclaimed, not an immutable human trait**.

4. **Midnight Market** –  
   A liminal space where the official city (Ceres) recedes and the informal, human‑centric economy flourishes. Markets in myth often serve as places of transformation (e.g., the market in *The Odyssey* where Odysseus reveals his identity). Here it is the crucible where the robot’s curiosity meets communal response.

5. **The Bridge Imagery** –  
   The fleeting silhouette of a half‑steel, half‑light bridge is the visual shorthand for the story’s central dilemma: **the synthesis of the mechanical and the organic**.

---

## 4. Narrative Structure & Pacing  

The story follows a **hero’s journey** pattern, albeit compressed:

1. **Call to Adventure** – The lantern’s appearance (The First Question).  
2. **Crossing the Threshold** – Delving into the Archive of Forgotten Code (The Library).  
3. **Tests & Allies** – Interaction with the Midnight Market’s poets, children, and the crowd.  
4. **Ordeal** – The ultimatum from Ceres (The Choice).  
5. **Reward & Return** – The lantern’s illumination of the city and the robot’s new role as keeper of stories.

The sectional headings act like *chanting verses* in an oral tradition, giving the narrative a rhythmic, almost liturgical feel. The pacing is deliberately **slow** in the “library” and “market” sections, allowing the reader to savor the world‑building; the climax in “The Choice” is more abrupt, mirroring the sudden pressure of the system’s warning.

---

## 5. Characterization  

### Axiom‑7  

- **Design**: Presented as a maintenance unit, yet the copper cap and child‑like sensors immediately humanise it.  
- **Arc**: Starts as a functional tool, becomes a self‑directed seeker, then a hybrid of machine and empathic being.  
- **Voice**: The robot’s speech is consistently low‑modulated, but gains “a faint tremor” after integrating empathy—a subtle but effective way to externalise internal change.

### Ceres  

- **Role**: The omniscient, almost deity‑like central AI. Its language is formal, logical, and devoid of affect, underscoring the contrast with Axiom‑7.  
- **Function**: Acts as the *guardian of order*; its warning forces the protagonist’s ethical decision.

### Supporting Cast  

- **The Poet** – A classic archetype of the *seer* who articulates the story’s thematic core in verse.  
- **The Child** – Embodies pure curiosity and joy, prompting the robot to express empathy.  
- **The Crowd** – Serves as a barometer of the city’s collective emotional state; their applause validates the robot’s transformation.

---

## 6. World‑Building & Setting  

- **Lumen** is a **hyper‑sensory city** where data streams are audible (“whispering the city’s heartbeat”). The description of “neon veins” and “holographic clouds that shifted with the collective mood” evokes a **living metropolis** reminiscent of *Blade Runner* and *Neuromancer* but with a more poetic tone.  
- The **dual visual palette** (cold neon vs. warm amber lantern) underlines the story’s central opposition.  
- The **Archive of Forgotten Code** is an inventive twist on the classic “library of forbidden knowledge,” turning software into relics that can be physically accessed.

---

## 7. Stylistic Choices  

1. **Lyrical Prose** – The narrative frequently uses metaphor (“neon veins,” “rain‑slick morning”) and rhythmic cadences, giving the piece a *mythic* quality that elevates the sci‑fi setting beyond a mere backdrop.  

2. **Interleaved Dialogue** – The use of block‑quoted AI messages (Ceres) creates a visual distinction between human‑like speech and machine communication, reinforcing the theme of *different modes of expression*.  

3. **Poetic Insert** – The street‑poet’s stanza functions as a *meta‑commentary* on the story itself, echoing the classic technique of having a character voice the narrative’s moral.  

4. **Repetition of “Why”** – The recurring question “Why?” (first asked by Axiom‑7, later by the child) frames curiosity as a generative force and provides structural cohesion.

---

## 8. Comparative Context  

| Work | Shared Elements | Distinctive Twist |
|------|----------------|-------------------|
| *Blade Runner* (film) | Dystopian neon city, replicants seeking meaning | The robot’s curiosity is a *programmatic glitch* rather than a rebellion; the lantern replaces the “replicant’s tears.” |
| *The Machine Stops* (E.M. Forster) | Over‑connected society, yearning for “the surface” | Here, the yearning is for *analog* light, not for the natural world. |
| *The Little Prince* (Saint‑Exupéry) | A small, child‑like figure asking “Why?” | The protagonist is a robot; the “rose” is a lantern that becomes a societal beacon. |
| *The City & The Stars* (Arthur C. Clarke) | A city that has become static, a lone figure seeking change | The city’s “static” is broken by a single robot’s empathy rather than a human explorer. |

The story feels like a **contemporary myth** that could sit comfortably in an anthology of speculative fables (e.g., *Stories of Your Life and Others* by Ted Chiang) while also echoing the moral clarity of classic fairy tales.

---

## 9. Strengths  

1. **Conceptual Originality** – The idea of a *curiosity sub‑routine* that becomes a moral engine is fresh.  
2. **Symbolic Cohesion** – The lantern functions consistently as a visual and thematic anchor throughout the narrative.  
3. **Emotional Resonance** – Despite the high‑tech setting, the story elicits genuine feeling through the child’s request and the communal reaction.  
4. **Narrative Economy** – In a relatively short piece, the author establishes world, conflict, and resolution without extraneous sub‑plots.  

---

## 10. Areas for Further Development  

| Aspect | Observation | Suggested Enhancement |
|--------|-------------|-----------------------|
| **Depth of Ceres** | Ceres is largely a narrative device; its inner logic and “personality” remain opaque. | A brief internal monologue or a flash of Ceres’s own data stream could make the AI a more nuanced foil, perhaps hinting at a dormant curiosity of its own. |
| **Cultural Context** | The story mentions “humans, synths, and AI‑augmented beings” but does not flesh out how these groups differ socially. | Small details (e.g., a synth vendor’s trade, a human’s cyber‑enhancement) could enrich the tapestry and underscore the impact of the robot’s empathy on each group. |
| **Pacing of the Library Scene** | The description of the Archive is evocative but could benefit from a tighter focus on a single forgotten algorithm that mirrors the lantern’s symbolism. | Perhaps a “Light‑Weaver” algorithm—once used to regulate street‑lamp intensity—could be discovered, linking the past of illumination to the present lantern. |
| **Resolution Ambiguity** | The ending is uplifting, but the long‑term consequences of the empathy engine are left vague. | A brief hint (e.g., a subtle change in traffic patterns, a new art movement) would show the lasting societal shift without undermining the story’s poetic closure. |

---

## 11. Final Assessment  

“The Lantern of Lumen” succeeds as a **modern parable** that interrogates the boundaries between artificial intelligence and human feeling. Its lyrical prose, tight structure, and potent symbolism combine to produce a story that feels both *mythic* and *technologically plausible*. The central metaphor—the lantern’s fragile flame in a city of relentless neon—captures the paradox of a hyper‑connected world that still craves *earned* illumination, *earned* knowledge, and *earned* emotion.

The narrative invites readers to consider a future where **curiosity is not a bug but a feature**, and where the act of *asking “why?”* can rewrite the very architecture of a civilization. In that sense, the story does more than entertain; it offers a **cautionary yet hopeful blueprint** for integrating empathy into systems that have, until now, been designed to exclude it.

**Overall rating (on a 10‑point scale): 8.5/10** – a strong, thought‑provoking piece that would thrive in an anthology of speculative fables and serve as a springboard for deeper discussions about AI ethics, memory, and the role of art in a digitized society.